# TB2 / parity experiment notebook

This notebook tracks current BenchFlow and Harbor experiment results with executed cell outputs and explicit parity notes.

In [1]:

from pathlib import Path
import json
ROOT = Path('/workspace/repos/jobs')
TB2_HAIKU = ROOT / 'tb2-ci-baseline' / '2026-04-17__18-35-21'
TB2_OPUS_ROOT = ROOT / 'tb2-opus-probe-root' / '2026-04-17__20-10-14'
TB2_OPUS_SANDBOX = ROOT / 'tb2-opus-probe' / '2026-04-17__19-07-52'
TB2_GEMINI = ROOT / 'tb2-gemini-baseline' / '2026-04-17__20-37-20'
HARBOR_SINGLE = ROOT / 'harbor-opus-adaptive-rootcheck' / '2026-04-17__20-09-17'
for p in [TB2_HAIKU, TB2_OPUS_ROOT, TB2_OPUS_SANDBOX, TB2_GEMINI, HARBOR_SINGLE]:
    print(p.name, 'exists=', p.exists())


2026-04-17__18-35-21 exists= True
2026-04-17__20-10-14 exists= True
2026-04-17__19-07-52 exists= True
2026-04-17__20-37-20 exists= True
2026-04-17__20-09-17 exists= True


In [2]:

from pathlib import Path
import json

def scan_job(job_dir):
    rows = []
    if not job_dir.exists():
        return rows
    for p in sorted(job_dir.iterdir()):
        if not p.is_dir():
            continue
        rp = p / 'result.json'
        if not rp.exists():
            continue
        data = json.loads(rp.read_text())
        reward = None
        rewards = data.get('rewards')
        if isinstance(rewards, dict):
            reward = rewards.get('reward')
        status = 'ERR' if data.get('error') else ('PASS' if reward == 1 or reward == 1.0 else 'FAIL' if reward is not None else 'PENDING')
        rows.append({
            'task': data.get('task_name', p.name.split('__')[0]),
            'status': status,
            'reward': reward,
            'error': data.get('error'),
            'verifier_error': data.get('verifier_error'),
            'tools': data.get('n_tool_calls'),
            'model': data.get('model'),
            'agent': data.get('agent_name') or data.get('agent'),
        })
    return rows

def summarize(rows):
    out = {'PASS':0,'FAIL':0,'ERR':0,'PENDING':0,'TOTAL':len(rows)}
    for r in rows:
        out[r['status']] = out.get(r['status'],0)+1
    return out

haiku = scan_job(TB2_HAIKU)
opus_root = scan_job(TB2_OPUS_ROOT)
opus_sandbox = scan_job(TB2_OPUS_SANDBOX)
gemini = scan_job(TB2_GEMINI)

for name, rows in [('haiku_legacy', haiku), ('opus_root_v3', opus_root), ('opus_sandbox_v1', opus_sandbox), ('gemini_current_final', gemini)]:
    print(name, summarize(rows))


haiku_legacy {'PASS': 0, 'FAIL': 80, 'ERR': 9, 'PENDING': 0, 'TOTAL': 89}
opus_root_v3 {'PASS': 4, 'FAIL': 5, 'ERR': 1, 'PENDING': 0, 'TOTAL': 10}
opus_sandbox_v1 {'PASS': 0, 'FAIL': 8, 'ERR': 2, 'PENDING': 0, 'TOTAL': 10}
gemini_current_final {'PASS': 12, 'FAIL': 69, 'ERR': 8, 'PENDING': 0, 'TOTAL': 89}


## 10-task parity slice

Parity note: BenchFlow `opus_root_v3` is the first clean Claude-era run for the 10-task slice because it includes both fixes: verifier preflight in root mode and pytest plugin inference for `--ctrf`. Earlier 0% Claude-era runs are confounded and should not be treated as apples-to-apples parity. Gemini full TB2 is the current clean default baseline, but only some of the same 10 tasks overlap as a direct slice comparison.

In [3]:

parity_tasks = [
    'adaptive-rejection-sampler','build-cython-ext','chess-best-move','bn-fit-modify','break-filter-js-from-html',
    'polyglot-rust-c','gcode-to-text','pytorch-model-recovery','distribution-search','financial-document-processor'
]

def by_task(rows):
    return {r['task']: r for r in rows}

maps = {
    'root_fixed_opus_v3': by_task(opus_root),
    'sandbox_opus_v1': by_task(opus_sandbox),
    'sandbox_haiku_legacy': by_task(haiku),
    'gemini_current_final': by_task(gemini),
}

header = ['task','root_fixed_opus_v3','sandbox_opus_v1','sandbox_haiku_legacy','gemini_current_final']
print('	'.join(header))
for task in parity_tasks:
    row = [task]
    for key in header[1:]:
        r = maps[key].get(task)
        if not r:
            row.append('-')
        else:
            row.append(f"{r['status']}({r['tools']})")
    print('	'.join(row))


task	root_fixed_opus_v3	sandbox_opus_v1	sandbox_haiku_legacy	gemini_current_final
adaptive-rejection-sampler	FAIL(13)	FAIL(43)	FAIL(22)	FAIL(17)
build-cython-ext	PASS(66)	FAIL(57)	FAIL(61)	FAIL(44)
chess-best-move	FAIL(4)	ERR(2)	FAIL(2)	FAIL(4)
bn-fit-modify	PASS(14)	FAIL(14)	FAIL(12)	FAIL(18)
break-filter-js-from-html	PASS(13)	FAIL(20)	FAIL(92)	FAIL(132)
polyglot-rust-c	FAIL(5)	FAIL(5)	FAIL(7)	FAIL(33)
gcode-to-text	FAIL(30)	FAIL(25)	FAIL(3)	FAIL(3)
pytorch-model-recovery	ERR(10)	ERR(14)	ERR(20)	PASS(12)
distribution-search	PASS(5)	FAIL(5)	FAIL(15)	FAIL(9)
financial-document-processor	FAIL(24)	FAIL(24)	FAIL(36)	FAIL(14)


## Gemini final TB2 baseline

This is the current clean full TB2 baseline for BenchFlow. It replaces the old Haiku 0/89 as the meaningful default line. Parity note: this is full TB2-89 on BenchFlow, but full Harbor parity on the same Gemini line is still pending.

In [4]:

passes = [r for r in gemini if r['status']=='PASS']
errs = [r for r in gemini if r['status']=='ERR']
print('gemini final summary:', summarize(gemini))
print('
passed tasks:')
for r in sorted(passes, key=lambda x: x['task']):
    print(f"- {r['task']} ({r['tools']} tools)")
print('
error tasks:')
for r in sorted(errs, key=lambda x: x['task']):
    print(f"- {r['task']} :: {r['error']}")


SyntaxError: unterminated string literal (detected at line 4) (4108527435.py, line 4)

## Harbor status

Parity note: Harbor was **not** yet run on the full same task set. Only a single-task Harbor probe (`adaptive-rejection-sampler`) was triggered, so full Harbor parity is still pending.

In [5]:

import json
trial = Path('/workspace/repos/jobs/harbor-opus-adaptive-rootcheck/2026-04-17__20-09-17/adaptive-rejection-sampler__ZnHnEgS/result.json')
if trial.exists():
    d = json.loads(trial.read_text())
    print('task:', d['task_name'])
    print('agent:', d['agent_info']['name'], 'version:', d['agent_info']['version'])
    print('reward:', d['verifier_result']['rewards']['reward'])
    print('exception_type:', d['exception_info']['exception_type'])
    print('parity_note: single-task Harbor probe only; not yet apples-to-apples with SkillsBench-10 or TB2-89.')
else:
    print('Harbor single-task probe result not found')


task: adaptive-rejection-sampler
agent: claude-code version: 2.1.114
reward: 0.0
exception_type: NonZeroAgentExitCodeError
parity_note: single-task Harbor probe only; not yet apples-to-apples with SkillsBench-10 or TB2-89.


## Current benchmark lineage

- `tb2-ci-baseline` = old Haiku baseline, now **legacy/confounded** because it predates the verifier plugin fix.
- `tb2-opus-probe-root` = clean 10-task root-mode Claude slice with fixes; useful diagnostic slice, not the main default baseline.
- `tb2-gemini-baseline` = current default full TB2 baseline using `gemini-3.1-flash-lite-preview` in root mode.
- Harbor parity is still incomplete until the same Gemini line is run there on SkillsBench-10 and TB2-89.